# kNN Hyperparameter Tuning & Evaluation

This notebook evaluates kNN on fused features (2067-dim).

Key finding: LDA dimensionality reduction (2067→6 dims) was critical.
Without LDA: F1-Macro = 0.24 | With LDA: F1-Macro = 0.77

Final model: n_neighbors=15, euclidean, uniform weights + LDA

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    f1_score
)
import warnings
warnings.filterwarnings('ignore')

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../saved_models', exist_ok=True)
print("Libraries loaded.")

Libraries loaded.


## Load Data

In [2]:
X_train = np.load('../data/processed/X_train_fused_optimized.npy')
y_train = np.load('../data/processed/y_train_optimized.npy')
X_test  = np.load('../data/processed/X_test_fused_optimized.npy')
y_test  = np.load('../data/processed/y_test_optimized.npy')

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

X_train shape: (8012, 2067)
X_test shape:  (2003, 2067)


## Normalize Features
kNN is distance-based — StandardScaler is required.
Without scaling, features with larger values dominate distance calculation.

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done.")
print("X_train_scaled shape:", X_train_scaled.shape)

# Save scaler
with open('../saved_models/scaler_fused.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved.")

Scaling done.
X_train_scaled shape: (8012, 2067)
Scaler saved.


## GridSearchCV — Hyperparameter Tuning
Testing:
- n_neighbors: [3, 5, 7, 11, 15]
- weights: [uniform, distance]
- metric: [euclidean, manhattan]

In [4]:
param_grid = {
    'n_neighbors': [3, 5, 7, 11, 15],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV for kNN...")
grid_knn.fit(X_train_scaled, y_train)

print("\nBest parameters:", grid_knn.best_params_)
print("Best F1-macro (CV):", round(grid_knn.best_score_, 4))

Starting GridSearchCV for kNN...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters: {'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'distance'}
Best F1-macro (CV): 0.9267


## LDA + kNN
- PCA maximizes variance but not class separability
- LDA directly maximizes class discrimination
- Reduces 2067 dimensions to n_classes-1 = 6 dimensions
- Explained variance ratios: [0.39, 0.22, 0.14, 0.12, 0.07, 0.06]

In [10]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# LDA uygula (max 6 component — 7 sınıf - 1)
lda = LinearDiscriminantAnalysis(n_components=6)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda  = lda.transform(X_test_scaled)

print(f"X_train_lda shape: {X_train_lda.shape}")
print(f"Explained variance ratio: {lda.explained_variance_ratio_}")

X_train_lda shape: (8012, 6)
Explained variance ratio: [0.38960377 0.2154609  0.14344504 0.11669107 0.07268401 0.06211521]


## GridSearchCV with LDA features

In [11]:
param_grid = {
    'n_neighbors': [3, 5, 7, 11, 15],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("param_grid ve cv_strategy tanımlandı.")

param_grid ve cv_strategy tanımlandı.


In [12]:
grid_knn_lda = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("Testing kNN with LDA...")
grid_knn_lda.fit(X_train_lda, y_train)

y_pred_lda = grid_knn_lda.best_estimator_.predict(X_test_lda)
y_proba_lda = grid_knn_lda.best_estimator_.predict_proba(X_test_lda)

acc_lda = accuracy_score(y_test, y_pred_lda)
f1_lda  = f1_score(y_test, y_pred_lda, average='macro')
auc_lda = roc_auc_score(y_test, y_proba_lda, multi_class='ovr', average='macro')

print("\nBest params:", grid_knn_lda.best_params_)
print(f"Accuracy : {acc_lda:.4f}")
print(f"F1-Macro : {f1_lda:.4f}")
print(f"ROC-AUC  : {auc_lda:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lda, target_names=CLASS_NAMES))

Testing kNN with LDA...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best params: {'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'uniform'}
Accuracy : 0.8597
F1-Macro : 0.7738
ROC-AUC  : 0.9056

Classification Report:
              precision    recall  f1-score   support

       akiec       0.64      0.66      0.65        65
         bcc       0.77      0.73      0.75       103
         bkl       0.76      0.67      0.71       220
          df       0.89      0.74      0.81        23
         mel       0.67      0.65      0.66       223
          nv       0.92      0.95      0.93      1341
        vasc       1.00      0.82      0.90        28

    accuracy                           0.86      2003
   macro avg       0.81      0.75      0.77      2003
weighted avg       0.86      0.86      0.86      2003



## Confusion Matrix — kNN + LDA

In [ ]:
cm_lda = confusion_matrix(y_test, y_pred_lda)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_lda, annot=True, fmt='d',
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            cmap='Blues')
plt.title(f'kNN + LDA — Confusion Matrix\n{grid_knn_lda.best_params_}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../outputs/knn_lda_fused_confusion_matrix.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Save best model

In [19]:
# LDA daha iyi sonuç verdi, bunu kaydedelim
import pickle

# Model kaydet
with open('../saved_models/knn_model_fused.pkl', 'wb') as f:
    pickle.dump(grid_knn_lda.best_estimator_, f)

with open('../saved_models/lda_transformer_fused.pkl', 'wb') as f:
    pickle.dump(lda, f)

print("Kaydedildi:")
print("- knn_model_fused.pkl (LDA ile en iyi model)")
print("- lda_transformer_fused.pkl")
print("- scaler.pkl")

Kaydedildi:
- knn_model_fused.pkl (LDA ile en iyi model)
- lda_transformer_fused.pkl
- scaler.pkl


<Figure size 640x480 with 0 Axes>

## Summary — kNN Results Comparison

| Method | Features | Accuracy | F1-Macro | ROC-AUC |
|--------|----------|----------|----------|---------|
| kNN + Scaling | Frozen ResNet (2048) | 0.47 | 0.24 | 0.70 |
| kNN + PCA | Frozen ResNet (2048) | 0.47 | 0.24 | — |
| kNN + LDA | Frozen ResNet (2048) | 0.62 | 0.46 | 0.82 |
| **kNN + LDA** | **Fine-tuned ResNet + Metadata (2067)** | **0.86** | **0.77** | **0.91** |

**Conclusion:** Fine-tuned ResNet-50 features with clinical metadata
fusion significantly improved kNN performance from F1=0.24 to F1=0.77.
LDA dimensionality reduction was critical in both feature spaces.
Best params: n_neighbors=15, euclidean, uniform weights.